In [7]:
using JuMP
using Gurobi          # or HiGHS, CPLEX, etc.
using CSV, DataFrames, Plots


# Unit Commitment Model

In [8]:
function build_pf_uc(pi_hat::Vector{Float64},
                     cf::Float64,
                     cstart::Float64,
                     cfixed::Float64,
                     Pmax::Float64,
                     Pmin::Float64,
                     R::Float64,
                     Mup::Int,
                     Mdown::Int,
                     y0::Int,
                     p0::Float64)

    T = length(pi_hat)

    model = Model(Gurobi.Optimizer)
    # set_silent(model)   # comment this out if you want solver output

    # Decision variables
    @variable(model, 0 <= p[1:T] <= Pmax)   # power output (MW)
    @variable(model, y[1:T], Bin)          # on/off
    @variable(model, u[1:T], Bin)          # startup
    @variable(model, w[1:T], Bin)          # shutdown

    # Objective: profit over full horizon (perfect foresight on pi_hat)
    @objective(model, Max,
        sum((pi_hat[t] - cf) * p[t] - cstart * u[t] - cfixed * y[t] for t in 1:T)
    )

    # Startup/shutdown logic
    for t in 1:T
        if t == 1
            # y1 - y0 = u1 - w1
            @constraint(model, y[1] - y0 == u[1] - w[1])
            @constraint(model, u[1] <= 1 - y0)
            @constraint(model, w[1] <= y0)
        else
            # yt - y(t-1) = ut - wt
            @constraint(model, y[t] - y[t-1] == u[t] - w[t])
            @constraint(model, u[t] <= 1 - y[t-1])
            @constraint(model, w[t] <= y[t-1])
        end
        @constraint(model, u[t] <= y[t])
        @constraint(model, w[t] <= 1 - y[t])
    end

    # Capacity limits and min stable load
    for t in 1:T
        @constraint(model, p[t] <= Pmax * y[t])
        @constraint(model, p[t] >= Pmin * y[t])
    end

    # Ramping (including from initial condition p0)
    @constraint(model,  p[1] - p0 <= R)
    @constraint(model,  p0 - p[1] <= R)
    for t in 2:T
        @constraint(model,  p[t] - p[t-1] <= R)
        @constraint(model,  p[t-1] - p[t] <= R)
    end

    # Minimum up-time: if you start at t, must stay on for at least Mup periods
    if Mup > 0
        for t in 1:(T - Mup + 1)
            @constraint(model, sum(y[k] for k in t:(t + Mup - 1)) >= Mup * u[t])
        end
    end

    # Minimum down-time: if you shut down at t, must stay off for at least Mdown periods
    if Mdown > 0
        for t in 1:(T - Mdown + 1)
            @constraint(model, sum(1 - y[k] for k in t:(t + Mdown - 1)) >= Mdown * w[t])
        end
    end

    return model, p, y, u, w
end



build_pf_uc (generic function with 1 method)

In [9]:
# -------------------------
# Example: full-year PF run
# -------------------------
# 1) Load your hourly prices (one row per hour, full year)
#    Say the CSV has a column :price in $/MWh

df = CSV.read("../clean_data/test_data_baseline.csv", DataFrame)
pi_hat = collect(df.HB_NORTH)

# 2) Set plant parameters (replace with your actual values)
cf     = 50.0        # $/MWh
cstart = 10_000.0    # $/start
cfixed = 2_000.0     # $/h
Pmax   = 171.0       # MW
Pmin   = 68.0        # MW
R      = 171.0       # MW per hour (or tighter if ramp binding)
Mup    = 4           # hours
Mdown  = 4           # hours
y0     = 0           # initially off
p0     = 0.0         # MW

model, p, y, u, w = build_pf_uc(pi_hat, cf, cstart, cfixed,
                                Pmax, Pmin, R,
                                Mup, Mdown,
                                y0, p0)

optimize!(model)

println("PF objective (total profit): ", objective_value(model))
p_opt = value.(p)
y_opt = value.(y)
u_opt = value.(u)
w_opt = value.(w)

T = length(p_opt)
t = 1:T

Set parameter Username
Set parameter LicenseID to value 2697103


ErrorException: Gurobi Error 10009: Version number is 13.0, license is for version 12.0

# Visualization

In [4]:
# Basic dispatch line
p_plot = plot(
    t, p_opt,
    xlabel = "Time period (t)",
    ylabel = "Power output p_t (MW)",
    label = "Dispatch",
    legend = :top,
    title = "Perfect-Foresight Dispatch"
)

# Mark startups (u_t = 1) and shutdowns (w_t = 1)
startup_idx   = findall(x -> x > 0.5, u_opt)
shutdown_idx  = findall(x -> x > 0.5, w_opt)

if !isempty(startup_idx)
    scatter!(
        p_plot,
        startup_idx,
        p_opt[startup_idx],
        markershape = :utriangle,
        label = "Startup",
    )
end

if !isempty(shutdown_idx)
    scatter!(
        p_plot,
        shutdown_idx,
        p_opt[shutdown_idx],
        markershape = :dtriangle,
        label = "Shutdown",
    )
end

display(p_plot)

y_plot = plot(
    t, y_opt,
    seriestype = :steppre,
    xlabel = "Time period (t)",
    ylabel = "Commitment y_t",
    yticks = ([0, 1], ["Off", "On"]),
    label = "On/Off",
    legend = :top,
    title = "Unit Commitment Schedule"
)

display(y_plot)

combined = plot(
    p_plot,
    y_plot,
    layout = (2, 1),
    size = (900, 700),
)

display(combined)


UndefVarError: UndefVarError: `t` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [5]:
# t index
T = length(pi_hat)
t = 1:T

# 1) Price + Dispatch on twin y-axes
plt1 = plot(
    t, pi_hat,
    xlabel = "Time period",
    ylabel = "Price (\$/MWh)",
    label  = "Price",
    legend = :top,
    title  = "Price vs Dispatch (Perfect Foresight)",
)

# Add dispatch on right y-axis
plot!(
    plt1,
    t, p_opt,
    ylabel = "Power (MW)",
    label  = "Dispatch",
    yaxis  = :right,
)

display(plt1)

# 2) Price with commitment shaded (0/1 scaled to MW or as band)

# Option A: show commitment as 0/1 line under price
plt2 = plot(
    t, pi_hat,
    xlabel = "Time period",
    ylabel = "Price (\$/MWh)",
    label  = "Price",
    legend = :top,
    title  = "Price with Commitment Overlay",
)

# Scale y_opt to [0, max(pi_hat)] just for visualization band
y_band = (maximum(pi_hat) * 0.15) .* y_opt  # 15% of max price

plot!(
    plt2,
    t, y_band,
    fillrange = 0,
    seriestype = :steppre,
    alpha = 0.3,
    label = "Committed (shaded when y_t = 1)",
)

display(plt2)


UndefVarError: UndefVarError: `p_opt` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# Test Unit Commitment against Actual Prices

In [6]:
"""
compute_profit(pi, p, y, u; cf, cstart, cfixed)

Compute realized profit for a *given* schedule (p, y, u) under a price path pi.

Arguments:
- pi::Vector{Float64}: price path ($/MWh), length T
- p::Vector{Float64}: dispatch (MW), length T
- y::Vector{Float64}: commitment (0/1), length T
- u::Vector{Float64}: startups (0/1), length T

Keyword args:
- cf::Float64: marginal cost ($/MWh)
- cstart::Float64: startup cost ($/start)
- cfixed::Float64: fixed cost ($/h)

Returns:
- profit::Float64
"""
function compute_profit(pi::Vector{Float64},
                        p::Vector{Float64},
                        y::Vector{Float64},
                        u::Vector{Float64};
                        cf::Float64,
                        cstart::Float64,
                        cfixed::Float64)

    T = length(pi)
    @assert length(p) == T && length(y) == T && length(u) == T "Length mismatch"

    profit = 0.0
    for t in 1:T
        margin   = (pi[t] - cf) * p[t]
        startup  = cstart * u[t]
        fixed    = cfixed * y[t]
        profit  += margin - startup - fixed
    end
    return profit
end


Base.Meta.ParseError: ParseError:
# Error @ /Users/eric/ERCOT_Peaker_Project/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:7:37
Arguments:
- pi::Vector{Float64}: price path ($/MWh), length T
#                                   └ ── identifier or parenthesized expression expected after $ in string

## Full Function with Optimization and Realization

In [ ]:
# ─────────────────────────────────────────────
# Assumes you have this from before:
# build_pf_uc(pi_hat, cf, cstart, cfixed, Pmax, Pmin, R, Mup, Mdown, y0, p0)
# ─────────────────────────────────────────────

"""
run_uc_forecast_vs_actual(pi_pred, pi_real; params...)

- pi_pred: predicted prices used in optimization (length T)
- pi_real: actual realized prices (same length T)

Keyword params:
- cf, cstart, cfixed, Pmax, Pmin, R, Mup, Mdown, y0, p0

Returns:
- forecast_profit_pred_prices::Float64   # objective when solved on predicted prices
- realized_profit_actual_prices::Float64 # same schedule evaluated on actual prices
- pf_profit_actual_prices::Float64       # perfect-foresight UC using actual prices
"""
function run_uc_forecast_vs_actual(pi_pred::Vector{Float64},
                                   pi_real::Vector{Float64};
                                   cf::Float64,
                                   cstart::Float64,
                                   cfixed::Float64,
                                   Pmax::Float64,
                                   Pmin::Float64,
                                   R::Float64,
                                   Mup::Int,
                                   Mdown::Int,
                                   y0::Int,
                                   p0::Float64)

    @assert length(pi_pred) == length(pi_real) "predicted and actual price vectors must have same length"
    T = length(pi_pred)

    # 1) Optimize UC using *predicted* prices
    model_forecast, p_f, y_f, u_f, w_f = build_pf_uc(
        pi_pred, cf, cstart, cfixed,
        Pmax, Pmin, R, Mup, Mdown, y0, p0
    )
    optimize!(model_forecast)

    p_forecast = value.(p_f)
    y_forecast = value.(y_f)
    u_forecast = value.(u_f)

    forecast_profit_pred_prices = objective_value(model_forecast)

    # 2) Evaluate that schedule under *actual* prices
    realized_profit_actual_prices = compute_profit(
        pi_real,
        p_forecast,
        y_forecast,
        u_forecast;
        cf = cf,
        cstart = cstart,
        cfixed = cfixed,
    )

    # 3) Solve perfect-foresight UC using *actual* prices as the objective
    model_pf, p_pf, y_pf, u_pf, w_pf = build_pf_uc(
        pi_real, cf, cstart, cfixed,
        Pmax, Pmin, R, Mup, Mdown, y0, p0
    )
    optimize!(model_pf)
    pf_profit_actual_prices = objective_value(model_pf)

    return forecast_profit_pred_prices,
           realized_profit_actual_prices,
           pf_profit_actual_prices
end
